# Characterization of the OLTC control schemes

In [ ]:
%load_ext autoreload
%autoreload 2

## 0 Imports and predefinitions

In [ ]:
# %matplotlib widget
# import ipympl

import numpy as np
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

parent = os.path.abspath("../..")
sys.path.insert(1, parent)
sys.path.append(
    str(
        os.path.dirname(os.path.dirname(os.path.abspath("oltc_characterization.ipynb")))
    )
)

from diffpssi.tools import *

from diffpssi.power_sim_lib.backend import *
from diffpssi.power_sim_lib.simulator import TestBench as Tb
from diffpssi.power_sim_lib.simulator import Recorder

from diffpssi.power_sim_lib.models.transformer import *
from diffpssi.power_sim_lib.models.voltage_controller import *
from diffpssi.power_sim_lib.models.blocks import *

## 1 Characterization of the OLTC scheme

### 1.1 Input functions

In [ ]:
def jump_input(t, a=1, b=1.1, threshold=1):

    # threshold = 1
    # a = 1
    # b = 0.8

    if isinstance(t, float):
        if t < threshold:
            jump = a * torch.ones((1, 1), dtype=torch.float64)
        else:
            jump = b * torch.ones((1, 1), dtype=torch.float64)
    else:  # isinstance(t, list):
        jump = []
        for i in t:
            if i < threshold:
                jump.append(a * torch.ones((1, 1), dtype=torch.float64))
            else:
                jump.append(b * torch.ones((1, 1), dtype=torch.float64))

    return jump

In [ ]:
def input_function(t, a=0.1, b=1.1):
    """
    Input function for the test bench simulation, respectively control block testing.
    """
    # a = 0.05

    a = a * torch.ones((1, 1), dtype=torch.float64)
    b = b * torch.ones((1, 1), dtype=torch.float64)

    # if isinstance(t, float):
    input = b - (b - 1) * torch.exp(-a * t) * torch.ones(
        (1, 1), dtype=torch.float64
    )
    # else:  # isinstance(t, list):
    #     input = []
    #     for i in t:
    #         input.append(
    #             b
    #             - (b - 1) * torch.exp(-a * i) * torch.ones((1, 1), dtype=torch.float64)
    #         )
    # # b - (b-1) *
    return input

### 1.2 Models to be Analyzed and Recorded

In [ ]:
analysis_models = [
    [OLTC_Discrete(t_1=5), 1],
    # [OLTC_Continuous(t_1=5), 1],
    # [OLTC_Continuous(t_1=10, db=0.05), 1],
    [FSM_Discrete(t_m=0.2, t_k=5, db=0.025), 1],
    [FSM_Discrete_2(t_m=0.2, t_k=5, db=0.025), 1],
]


def record_dict(simulation, call=False):
    record_dict = {
        "discrete OLTC model": simulation.diff_models[0].u_l,
        # 'continuous OLTC model':    simulation.diff_models[1].u_l,
        "discrete FSM model": simulation.diff_models[1].u_l,
        "discrete FSM model 2": simulation.diff_models[2].u_l,
    }
    if call:
        return record_dict.values()
    else:
        return record_dict

### 1.3 Run characterization and Data processing

In [ ]:
tb = Tb(
    time_step=0.005,
    sim_time=40,
    inspection_models=analysis_models,
    input_func=input_function,
    record_func=record_dict,
)

tb.diff_models[0].dir = -1
tb.diff_models[1].dir = -1
tb.diff_models[2].dir = -1

# t, recorder = tb.run_oltc_control()
t, recorder = tb.run_oltc_control()

record_list = tb.record_list()

## 2 Plotting and export

In [ ]:
x_i = torch.swapaxes(tb.input_func(torch.tensor(t)), 0, 1)
# x_i = tb.input_func(t)
output = recorder * x_i

In [ ]:
# Format shall be [batch, timestep, value]
plt.figure()
for i in range(len(record_list)):  # np.shape(analysis_models)[0]):
    plt.plot(t, output[0, :, i], label=record_list[i])

plt.plot(t, x_i[:], label="input function")

# plt.plot(t, recorder[0, :, 0], label=f'output model OLTC discrete')

plt.legend(loc="upper right")
plt.grid()
plt.xlabel("Time (s)")
plt.ylabel("Voltage (p.u.)")
# plt.savefig('./data/oltc_control_characterization_jump.pdf')
plt.show()

In [ ]:
# Format shall be [batch, timestep, value]
plt.figure()

plt.plot(t, output[0, :, 0], label=record_list[0])

plt.plot(t, x_i[:], label="input function")
plt.hlines(1.05, xmin=0, xmax=40, linestyles="dashed", label="Deadband")

# plt.plot(t, recorder[0, :, 0], label=f'output model OLTC discrete')

plt.legend()  # loc='upper right')
plt.grid()
plt.xlabel("Time (s)")
plt.ylabel("Voltage (p.u.)")
plt.savefig("./data/oltc_control_characterization_presentation.pdf")
plt.show()